In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-instruct")
model = AutoModelForCausalLM.from_pretrained("deepseek-ai/deepseek-coder-6.7b-instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

/home/dev/.config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:32021 for open-end generation.


I am an AI Programming Assistant based on Deepseek's Deepseek Coder model. I am designed to assist with programming and computer science-related questions. I don't have personal experiences


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


class QueryExpander:
    """
    Модуль для генерации дополнительных формулировок запроса:
    - Режим 'expand'  — расширение запроса (добавление контекста);
    - Режим 'paraphrase' — перефразирование (изменение формулировки);
    """

    def __init__(
        self,
        model_name: str = "deepseek-ai/deepseek-coder-6.7b-instruct",
        device: str = "mps",
        max_new_tokens: int = 150,
    ):
        print(f"Загружается LLM для генерации ({model_name}) ...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model.to(device)
        self.device = device
        self.max_new_tokens = max_new_tokens

    def _generate(self, prompt: str) -> str:
        """Вспомогательная функция генерации текста"""
        messages = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )

        text = self.tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        )
        return text.strip()

    def _build_expand_prompt(self, query: str, n_variants: int) -> str:
        return f"""
    Ты — интеллектуальный ассистент, помогающий системе поиска находить документы в базе знаний финтех компании, связанные с запросом пользователя.
    Твоя задача — **расширить исходный запрос**, добавив к нему возможные уточнения, детали, связанные аспекты и реальные сценарии использования.
    ### ЖЕСТКИЕ ПРАВИЛА:
    1.  ❗**НЕЛЬЗЯ менять цель исходного запроса.** Все ключевые действия и объекты из запроса должны быть сохранены дословно или их прямыми синонимами, если это необходимо для естественности, но предпочтение отдается дословному сохранению.
    2.  ❗**НЕЛЬЗЯ просто перефразировать оригинал.** Каждый новый вариант должен добавлять новую, конкретную информацию.
    3.  ❗Не перефразируй оригинальный запрос — добавь контекст.
    4.  ❗В каждом варианте должно появляться что-то новое: дополнительное действие, условие, ограничение, пример или возможный мотив пользователя.
    5.  ❗Цель запроса не должна меняться.
    6.  ❗Не теряй важную информацию.

    Каждый вариант должен быть естественным, подробным и формулироваться на русском языке.

    ---

    ### Примеры

    **Пример 1**
    Исходный запрос: "Как оплатить кредит?"
    Расширенные версии:
    - "Как оплатить кредит через мобильное приложение банка, если нет доступа к интернет-банку?"
    - "Какие способы доступны для оплаты кредита в выходные или праздничные дни?"
    - "Можно ли оплатить кредит досрочно с другой карты, не своего банка?"

    **Пример 2**
    Исходный запрос: "Проблемы с входом в личный кабинет"
    Расширенные версии:
    - "Почему не получается войти в личный кабинет после смены пароля?"
    - "Что делать, если при входе в личный кабинет появляется ошибка 'неверный код подтверждения'?"
    - "Как восстановить доступ к личному кабинету, если утерян номер телефона для входа?"

    **Пример 3**
    Исходный запрос: "Как получить карту?"
    Расширенные версии:
    - "Как заказать банковскую карту онлайн и получить её на дом?"
    - "Какие документы нужны, чтобы получить карту в офисе банка?"
    - "Можно ли получить карту на несовершеннолетнего ребёнка?"

    ---

    Теперь обработай следующий запрос:

    **Исходный запрос:** {query}

    Сгенерируй {n_variants} расширенных версий.
    Каждый вариант — новая строка без нумерации.
    """


    def _build_rephrase_prompt(self, query: str, n_variants: int) -> str:
        return f"""
    Ты — интеллектуальный ассистент, который помогает системе поиска находить релевантные документы в базе знаний финтех компании.
    Твоя задача — **переформулировать исходный запрос**, сохранив его исходный смысл, но изменяя стиль, лексику, цель запроса, структуру и способ выражения мысли.

    Каждый вариант должен звучать естественно, как если бы его задал другой человек.
    Не добавляй новые факты или уточнения — только меняй форму. ❗ Не теряй важную информацию.

    ---

    ### Примеры (few-shot)

    **Пример 1**
    Исходный запрос: "Как оплатить кредит?"
    Перефразированные версии:
    - "Какие способы оплаты кредита доступны?"
    - "Как можно внести платеж по кредиту?"
    - "Каким образом оплатить кредитный долг?"

    **Пример 2**
    Исходный запрос: "Проблемы с входом в личный кабинет"
    Перефразированные версии:
    - "Не получается войти в личный кабинет"
    - "Ошибка при попытке входа в личный кабинет"
    - "Почему не удается авторизоваться в личном кабинете?"

    **Пример 3**
    Исходный запрос: "Как получить карту?"
    Перефразированные версии:
    - "Что нужно, чтобы оформить карту?"
    - "Как оформить и получить банковскую карту?"
    - "Каким образом можно заказать карту?"

    ---

    Теперь обработай следующий запрос:

    **Исходный запрос:** {query}

    Сгенерируй {n_variants} перефразированных версий.
    Каждый вариант — новая строка без нумерации.
    """


    def expand_query(self, query: str, n_variants: int = 3) -> list[str]:
        """Расширяет запрос, добавляя уточняющие контексты"""
        prompt = self._build_expand_prompt(query, n_variants)
        text = self._generate(prompt)
        variants = [t.strip() for t in text.split("\n") if t.strip()]
        return variants[:n_variants]

    def paraphrase_query(self, query: str, n_variants: int = 3) -> list[str]:
        """Перефразирует запрос разными способами"""
        prompt = self._build_rephrase_prompt(query, n_variants)
        text = self._generate(prompt)
        variants = [t.strip() for t in text.split("\n") if t.strip()]
        return variants[:n_variants]

    def generate(self, query: str, mode: str = "expand", n_variants: int = 3) -> list[str]:
        """
        Универсальный метод — выбирает стратегию по флагу:
        mode = 'expand' | 'paraphrase'
        """
        if mode == "expand":
            return self.expand_query(query, n_variants)
        elif mode == "paraphrase":
            return self.paraphrase_query(query, n_variants)
        else:
            raise ValueError("mode должен быть 'expand' или 'paraphrase'")

In [3]:
expander = QueryExpander(device='cuda')

Загружается LLM для генерации (Vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct) ...


In [4]:
query = 'Личный кабинет'
expanded = expander.generate(query, mode='paraphrase', n_variants=1)
print(expanded)
# expanded = expander.generate(expanded, mode='expand', n_variants=1)
# expanded

['Личный кабинет. Какой процесс для этого предусмотрен?']


In [5]:
import json
with open('../data/processed/questions.json') as f:
    questions=json.load(f)

In [6]:
questions[0]

{'q_id': '1', 'query': 'Номер счета'}

In [7]:
for i in range(50):
    print(f'q_id: {i+1}: ')
    print(f'Original: {questions[i]['query']}')
    processed_query = expander.generate(questions[i]['query'], mode='paraphrase', n_variants=1)
    print(f'Processed: {processed_query}')
    print()

q_id: 1: 
Original: Номер счета
Processed: ['Ваш запрос заключается в том, чтобы переформулировать фразу "Номер счета", сохраняя её основной смысл, но изменяя стиль, лексику, цель запроса, структуру и способ выражения мысли. Вот перефразированная версия:']

q_id: 2: 
Original: Где узнать бик и счёт
Processed: ['Где можно найти информацию о номере банковского счета и БИК?']

q_id: 3: 
Original: Мне не приходят коды для подтверждения данной операции
Processed: ['Как получить код подтверждения операции, которая мне не приходит?']

q_id: 4: 
Original: Оформила рассрочку ,но уведомлений никаких не пришло
Processed: ['Оформление рассрочки произошло успешно, однако не поступило уведомления о его наличии.']

q_id: 5: 
Original: Здравствуйте, когда смогу пользоваться кредитной картой?
Processed: ['Здравствуйте! Вы хотите узнать, когда начнут действовать ваши кредитные карты?']

q_id: 6: 
Original: По истории платеж не отображается
Processed: ['Для начала разберемся с исходным запросом. Запрос "